In [ ]:
##### Bibliotecas
import os
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, models, transforms
import matplotlib.pyplot as plt

# Ignite
from ignite.engine import Engine, Events
from ignite.handlers import EarlyStopping
from ignite.metrics import Accuracy, Loss

# Optuna
import optuna

# Organização do dataset
data = "/home/jovyan/DADOS-DIVIDIDOS"
feature_extract = True

In [ ]:
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(30),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2),
        transforms.RandomVerticalFlip(),
        transforms.RandomPerspective(distortion_scale=0.2, p=0.5, interpolation=3, fill=0),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(224),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'test': transforms.Compose([
        transforms.Resize(224),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
}

image_datasets = {x: datasets.ImageFolder(os.path.join(data, x), data_transforms[x]) for x in ['train', 'val', 'test']}

In [ ]:
# Função para ajustar gradientes
def set_parameter_requires_grad(model, feature_extracting):
    if feature_extracting:
        for param in model.parameters():
            param.requires_grad = False
            
            
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet101(pretrained=True)
set_parameter_requires_grad(model, feature_extract)

model_resnet = "/home/jovyan/models/ResNet_101_ImageNet_plant-model-84.pth"
state_dict = torch.load(model_resnet)

state_dict.pop('fc.weight')
state_dict.pop('fc.bias')

model.load_state_dict(state_dict, strict=False)

num_features = model.fc.in_features
model.to(device)


In [ ]:
# Função de treinamento e validação
def train_step(engine, batch):
    x, y = batch
    x, y = x.to(device), y.to(device)

    model.train()
    y_pred = model(x)
    loss = criterion(y_pred, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item()

def validation_step(engine, batch):
    model.eval()
    with torch.no_grad():
        x, y = batch
        x, y = x.to(device), y.to(device)
        y_pred = model(x)
        return y_pred, y

# Função `objective` para busca bayesiana com Optuna
def objective(trial):
    global model, optimizer, criterion
    
    # Hiperparâmetros
    dropout_rate1 = trial.suggest_uniform("dropout1", 0.2, 0.5)
    dropout_rate2 = trial.suggest_uniform("dropout2", 0.2, 0.5)
    num_neurons_fc1 = trial.suggest_categorical("num_neurons_fc1", [256, 512, 1024])
    num_neurons_fc2 = trial.suggest_categorical("num_neurons_fc2", [128, 256, 512])
    activation = trial.suggest_categorical("activation", ["ReLU", "LeakyReLU", "Tanh"])
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])
    optimizer_name = trial.suggest_categorical("optimizer", ["SGD", "Adam"])
    lr = trial.suggest_loguniform("lr", 1e-5, 1e-2)
    momentum = trial.suggest_uniform("momentum", 0.7, 0.99) if optimizer_name == "SGD" else None

    # Função de ativação
    activation_fn = getattr(nn, activation)()

    # Redefinir o DataLoader com o batch size sugerido
    dataloaders_dict = {
        'train': torch.utils.data.DataLoader(image_datasets['train'], batch_size=batch_size, shuffle=True),
        'val': torch.utils.data.DataLoader(image_datasets['val'], batch_size=batch_size, shuffle=False)
    }

    # Configurar o modelo com os hiperparâmetros sugeridos
    model.fc = nn.Sequential(
        nn.Dropout(p=dropout_rate1),
        nn.Linear(num_features, num_neurons_fc1),
        activation_fn,
        nn.Dropout(p=dropout_rate2),
        nn.Linear(num_neurons_fc1, num_neurons_fc2),
        activation_fn,
        nn.Linear(num_neurons_fc2, 2)
    )
    model.to(device)

    # Configurar o otimizador
    params_to_update = [p for p in model.parameters() if p.requires_grad]
    if optimizer_name == "SGD":
        optimizer = optim.SGD(params_to_update, lr=lr, momentum=momentum)
    else:
        optimizer = optim.Adam(params_to_update, lr=lr)

    # Definir função de perda
    criterion = nn.CrossEntropyLoss()

    # Ignite trainers
    trainer = Engine(train_step)
    evaluator = Engine(validation_step)

    # Métricas
    val_metrics = {
        "accuracy": Accuracy(),
        "loss": Loss(criterion)
    }
    for name, metric in val_metrics.items():
        metric.attach(evaluator, name)

    @trainer.on(Events.EPOCH_COMPLETED)
    def log_training_results(engine):
        evaluator.run(dataloaders_dict['val'])
        metrics = evaluator.state.metrics
        print(f"Val Accuracy: {metrics['accuracy']:.4f}")
        
        # Early Stopping
        score_function = lambda engine: engine.state.metrics['accuracy']
        handler = EarlyStopping(patience=10, score_function=score_function, trainer=trainer)
        evaluator.add_event_handler(Events.COMPLETED, handler)

    # Executar o treinamento
    trainer.run(dataloaders_dict['train'], max_epochs=100)

    # Obter a acurácia final
    evaluator.run(dataloaders_dict['val'])
    return evaluator.state.metrics["accuracy"]

# Rodar o estudo Optuna
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

# Melhor conjunto de hiperparâmetros
print("Best trial:")
trial = study.best_trial
print(f"Accuracy: {trial.value}")
print("Best hyperparameters: ", trial.params)